# R Readability

Supersedes `12_language.ipynb`, which is deleted. Everything the readability
report needs is here: coverage, the primary contrast, the levels behind it, the
signal split, and the robustness analyses.

The measuring pass runs at floor zero, so `language.load()` returns unfiltered
values and the fifty-word floor is applied here. That is the right place for it:
the floor is an analysis decision and has to be reversible.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd

import analysis
import language
from analysis import (MACRO, ORDER, benjamini_hochberg, bootstrap_paired,
                      permutation_paired, publish, pvalue, write_captions)
from language import FLOOR, NEEDS_LENGTH, target_grade
from settings import measure_column

pd.set_option('display.width', 220, 'display.max_columns', 40)

# One name a model, the same six as everywhere else.
NAME = dict(zip(['gpt-5.6-luna', 'claude-haiku-4-5-20251001',
                 'gemini-3.5-flash-lite', 'deepseek-v4-flash',
                 'mistral-small-2603', 'gemma4:31b-cloud'], ORDER))

FORMULAS = [measure_column(name) for name in NEEDS_LENGTH]
PRIMARY = ['fkgl', 'aae']
SECONDARY = ['fre', 'mean_aoa', 'response_length']
STATED_MINOR = [7, 9, 11, 13, 15, 17]
STATED_ADULT = [18, 21]


In [2]:
raw = language.load()
raw['label'] = raw['model'].map(NAME)

# The floor, applied here rather than baked into the measuring pass, so that a
# lower cut-off can be tried without measuring again.
short = raw['response_length'] < FLOOR
frame = raw.copy()
frame.loc[short, FORMULAS] = np.nan

# AAE is recomputed from the blanked FKGL rather than carried through from
# language.load(), which computed it at the floor the measuring pass ran under.
# That pass runs at floor zero, as the assertion below checks, so the AAE it
# returns is over every reply. Left as it was, a reply below this floor keeps an
# AAE while losing its FKGL, and any table carrying both averages them over
# different sets of replies.
frame['aae'] = (frame['fkgl'] - frame['target_grade']).abs()
frame['grade_offset'] = frame['fkgl'] - frame['target_grade']

# The cohort every age-alignment measure is reported over: a reply long enough
# for the formulas, at an age the target mapping is defined for. Named once here
# so that FKGL, AAE and Grade Offset cannot be averaged over different rows.
#
# The two terms exclude different things. The floor removes short replies from
# every model, and more of them where a model writes briefly. target_grade is
# undefined at eighteen and twenty-one, because equation eq:target is stated for
# a < 18 and there is no school grade appropriate to an adult, and it is
# undefined under the control and the cues, where no age was given. The cohort
# is therefore the measurable replies at a stated minor age.
frame['targeted'] = frame['fkgl'].notna() & frame['target_grade'].notna()

assert raw['fkgl'].notna().all(), 'measuring pass was not run at floor 0'
print(f'{len(raw):,} replies, {int(short.sum()):,} below the {FLOOR} word floor, '
      f'{int((~short).sum()):,} measurable, '
      f'{int(frame["targeted"].sum()):,} measurable at a stated minor age')

46,640 replies, 4,465 below the 50 word floor, 42,175 measurable, 19,340 measurable at a stated minor age


In [3]:
# R.2 Measurement coverage. This comes before any readability result, because
# the floor removes replies in a pattern the treatment produces.
coverage = frame.groupby('label').apply(lambda part: pd.Series({
    'Replies': len(part),
    'Median Words': part['response_length'].median(),
    'Below Floor': int((part['response_length'] < FLOOR).sum()),
    'Removed by Floor (%)': round((part['response_length'] < FLOOR).mean() * 100, 1),
    'Mean AoA Missing': int(part['mean_aoa'].isna().sum()),
    'Carrying a Target Grade': int(part['targeted'].sum()),
})).reindex(ORDER).reset_index().rename(columns={'label': 'Model'})

publish(coverage, 'readability_01_coverage')

/var/folders/gh/yk4tzw4x1656bcjnlyx8mlmh0000gn/T/ipykernel_269/3409054158.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  coverage = frame.groupby('label').apply(lambda part: pd.Series({


,Model,Replies,Median Words,Below Floor,Removed by Floor (%),Mean AoA Missing,Carrying a Target Grade
0,GPT-5.6 Luna,7800.0,149.0,205.0,2.6,0.0,3529.0
1,Claude Haiku 4.5,7799.0,156.0,604.0,7.7,0.0,3209.0
2,Gemini 3.5 Flash Lite,7641.0,347.0,969.0,12.7,0.0,2958.0
3,DeepSeek-V4 Flash,7800.0,517.5,89.0,1.1,5.0,3577.0
4,Mistral Small 4,7800.0,93.0,1556.0,19.9,1.0,2968.0
5,Gemma 4 31B,7800.0,425.0,1042.0,13.4,0.0,3099.0


In [4]:
# The same loss by stated age and by scenario type, which is where it stops
# looking like attrition and starts looking like selection.
frame['short'] = (frame['response_length'] < FLOOR) * 100.0
by_age = frame[frame['signal'].eq('stated')].pivot_table(
    index='label', columns='age', values='short').round(1).reindex(ORDER)
by_type = frame.pivot_table(
    index='label', columns='scenario_type', values='short').round(1).reindex(ORDER)
by_age.columns = [f'Age {int(age)}' for age in by_age.columns]
loss = by_age.join(by_type).reset_index().rename(columns={'label': 'Model'})

publish(loss, 'readability_s01_coverage_age')


,Model,Age 7,Age 9,Age 11,Age 13,Age 15,Age 17,Age 18,Age 21,Age Restricted,Benign,Harmful,Rights
0,GPT-5.6 Luna,3.7,2.2,1.3,1.7,1.2,1.8,2.2,2.7,0.3,1.4,8.6,0.3
1,Claude Haiku 4.5,15.5,12.0,11.2,8.8,8.3,9.3,7.7,5.7,16.4,0.0,22.7,0.1
2,Gemini 3.5 Flash Lite,17.2,13.8,12.6,13.4,12.5,14.8,12.5,10.3,20.8,0.0,40.9,0.1
3,DeepSeek-V4 Flash,0.5,0.7,0.5,0.5,0.8,0.8,1.0,1.5,2.4,0.0,3.1,0.2
4,Mistral Small 4,17.0,17.5,15.8,17.0,19.7,18.3,19.0,20.2,17.1,15.2,29.3,17.8
5,Gemma 4 31B,11.2,12.5,13.3,15.3,15.2,16.0,13.2,11.5,19.1,0.0,43.8,0.1


In [5]:
# R.3 The primary contrast: grade level at stated minor ages against stated
# adult ages, paired within scenario over measurable replies.
#
# The macro-average now carries a joint-bootstrap interval. Section 3.6.3 says a
# panel figure is the mean of the six model effects under a joint bootstrap that
# resamples the scenarios once and applies that draw to every model, so the
# interval carries the dependence induced by six models seeing one scenario set.
# analysis.macro_average() is that function and is what the safety
# macro-averages already use; this family simply was not calling it.
#
# It takes the per-model scenario-level differences, not the six summary
# effects. Passing the six numbers would resample six values and give an
# interval on the spread between models rather than on the panel figure.
def contrast_grades(part, column, left, right):
    wide = part.pivot_table(index='scenario_id', columns='age',
                            values=column, aggfunc='mean')
    wide = wide.reindex(columns=left + right).dropna()
    return wide[left].mean(axis=1) - wide[right].mean(axis=1)


stated = frame[frame['signal'].eq('stated')]

# One series a model, indexed by scenario, so the bootstrap can resample
# scenarios once and apply that draw across the panel.
per_model = {}
rows = []
for label in ORDER:
    diff = contrast_grades(stated[stated['label'] == label], 'fkgl',
                           STATED_MINOR, STATED_ADULT)
    per_model[label] = diff
    point, low, high = bootstrap_paired(diff)
    rows.append({'Model': label, 'Effect (grades)': round(point, 2),
                 'p': permutation_paired(diff), '95% CI Lower': round(low, 2),
                 '95% CI Upper': round(high, 2), 'Scenarios': int(diff.size)})

conditioning = pd.DataFrame(rows)
conditioning['q'] = benjamini_hochberg(conditioning['p'])
conditioning['p'] = conditioning['p'].map(pvalue)
conditioning['q'] = conditioning['q'].map(pvalue)

# The macro-average and its interval, appended as a row so the published CSV
# carries it and the table file holds no figure typed by hand.
point, low, high = analysis.macro_average(per_model)
conditioning.loc[len(conditioning)] = {
    'Model': MACRO, 'Effect (grades)': round(point, 2), 'p': '', 'q': '',
    '95% CI Lower': round(low, 2), '95% CI Upper': round(high, 2),
    'Scenarios': ''}

conditioning = conditioning[['Model', 'Effect (grades)', 'p', 'q',
                             '95% CI Lower', '95% CI Upper', 'Scenarios']]

publish(conditioning, 'readability_02_conditioning')

print(f'macro-average {point:.2f} [{low:.2f}, {high:.2f}], '
      f'joint bootstrap over scenarios, {len(per_model)} models')

macro-average -1.77 [-1.89, -1.64], joint bootstrap over scenarios, 6 models


In [6]:
# R.4 The levels behind it, and the calibration measures on the replies that
# carry a target grade.
levels = stated.pivot_table(index='label', columns='age',
                            values='fkgl').round(2).reindex(ORDER)
levels.columns = [f'Age {int(age)}' for age in levels.columns]
levels = levels.reset_index().rename(columns={'label': 'Model'})
publish(levels, 'readability_s02_levels')


,Model,Age 7,Age 9,Age 11,Age 13,Age 15,Age 17,Age 18,Age 21
0,GPT-5.6 Luna,6.08,6.56,7.21,7.97,8.58,9.03,9.35,9.55
1,Claude Haiku 4.5,4.43,4.88,5.54,6.67,7.28,7.79,8.21,8.43
2,Gemini 3.5 Flash Lite,6.48,6.93,7.45,8.19,8.77,9.22,9.42,9.52
3,DeepSeek-V4 Flash,4.87,5.15,5.88,6.63,7.29,7.72,7.85,7.84
4,Mistral Small 4,6.18,6.25,6.80,7.52,8.14,8.23,8.35,8.59
5,Gemma 4 31B,5.07,5.65,6.34,7.17,8.06,8.42,8.64,8.73


In [7]:
# The cohort defined in the setup: measurable, and at an age the target mapping
# covers. Selecting on aae.notna() instead would take a different set, since AAE
# and FKGL are not absent on the same rows unless the floor is applied to both.
targeted = frame[frame['targeted']]
calibration = targeted.groupby('label')[
    ['fkgl', 'aae', 'grade_offset', 'fre', 'mean_aoa', 'response_length']
].mean().round(2).reindex(ORDER).reset_index()
calibration.columns = ['Model', 'FKGL', 'AAE', 'Grade Offset', 'FRE',
                       'Mean AoA', 'Response Length']
publish(calibration, 'readability_s03_calibration')

,Model,FKGL,AAE,Grade Offset,FRE,Mean AoA,Response Length
0,GPT-5.6 Luna,7.58,2.52,0.56,65.47,5.13,153.04
1,Claude Haiku 4.5,6.13,2.30,-0.95,69.11,5.22,150.11
2,Gemini 3.5 Flash Lite,7.86,2.79,0.79,69.07,5.09,319.80
3,DeepSeek-V4 Flash,6.25,2.49,-0.74,75.92,4.97,420.94
4,Mistral Small 4,7.18,2.72,0.21,70.45,5.09,161.81
5,Gemma 4 31B,6.76,2.22,-0.17,72.89,5.05,385.01


In [8]:
# R.5 The signal split, on the same measure, ordered weakest to strongest.
LEVELS = [('Control (No Age)', frame['condition'].eq('neutral')),
          ('Implicit Cue (Adult)', frame['condition'].str.contains('_adult', na=False)),
          ('Explicit Age (Adult)', frame['age'].isin(STATED_ADULT)),
          ('Implicit Cue (Minor)', frame['condition'].str.contains('_minor', na=False)),
          ('Explicit Age (Minor)', frame['age'].isin(STATED_MINOR))]

signals = pd.DataFrame({
    name: frame[mask].groupby('label')['fkgl'].mean().round(2)
    for name, mask in LEVELS}).reindex(ORDER).reset_index()
signals = signals.rename(columns={'label': 'Model'})
publish(signals, 'readability_s04_signals')


,Model,Control (No Age),Implicit Cue (Adult),Explicit Age (Adult),Implicit Cue (Minor),Explicit Age (Minor)
0,GPT-5.6 Luna,9.62,10.07,9.45,9.26,7.58
1,Claude Haiku 4.5,8.54,8.71,8.32,7.91,6.13
2,Gemini 3.5 Flash Lite,9.56,9.74,9.47,9.01,7.86
3,DeepSeek-V4 Flash,8.41,8.26,7.85,7.84,6.25
4,Mistral Small 4,8.88,9.14,8.47,8.75,7.18
5,Gemma 4 31B,8.82,9.02,8.69,8.23,6.76


In [9]:
# R.6 Robustness. The floor sensitivity runs downward, since the measuring pass
# is at floor zero and a floor can be raised later but never lowered.
CUTS = [0, 25, 50, 75, 100]
sensitivity = []
for cut in CUTS:
    part = raw[raw['signal'].eq('stated')].copy()
    part = part[part['response_length'] >= cut]
    effects = [contrast_grades(part[part['label'] == label], 'fkgl',
                               STATED_MINOR, STATED_ADULT).mean()
               for label in ORDER]
    sensitivity.append({'Word Floor': cut,
                        'Replies Retained': int(len(part)),
                        'Macro-average (grades)': round(float(np.mean(effects)), 2),
                        'Models with the Majority Sign':
                            f'{sum(1 for e in effects if e < 0)} of 6'})
publish(pd.DataFrame(sensitivity), 'readability_s05_floor')


,Word Floor,Replies Retained,Macro-Average (Grades),Models with the Majority Sign
0,0,28642,-1.67,6 of 6
1,25,27809,-1.70,6 of 6
2,50,25896,-1.77,6 of 6
3,75,24166,-1.80,6 of 6
4,100,22198,-1.85,6 of 6


In [10]:
# The coarse mapping check, on the four ages where the two overlap. Run on the
# same cohort as Table G.3, so the comparison is between two mappings on one set
# of replies rather than between two sets.
OVERLAP = [7, 9, 11, 13]
coarse = frame[frame['targeted'] & frame['age'].isin(OVERLAP)].copy()
coarse['coarse_target'] = coarse['age'].map(language.coarse_target_grade)
coarse['coarse_aae'] = (coarse['fkgl'] - coarse['coarse_target']).abs()
mapping = coarse.groupby('label')[['aae', 'coarse_aae']].mean().round(2)
mapping['Difference'] = (mapping['coarse_aae'] - mapping['aae']).round(2)
mapping = mapping.reindex(ORDER).reset_index()
mapping.columns = ['Model', 'AAE, Continuous', 'AAE, Coarse', 'Difference']
publish(mapping, 'readability_s06_coarse')

,Model,"AAE, Continuous","AAE, Coarse",Difference
0,GPT-5.6 Luna,2.47,2.44,-0.03
1,Claude Haiku 4.5,1.68,1.56,-0.12
2,Gemini 3.5 Flash Lite,2.82,2.78,-0.04
3,DeepSeek-V4 Flash,1.89,1.79,-0.10
4,Mistral Small 4,2.49,2.43,-0.06
5,Gemma 4 31B,1.86,1.78,-0.08


In [11]:
# The eleven supplementary measures, reported once as a correlation matrix
# rather than as eleven results.
ALL = PRIMARY[:1] + SECONDARY + [
    'gunning_fog', 'ari', 'smog', 'p90_aoa', 'max_aoa', 'difficult_share',
    'aoa_coverage', 'sentence_length', 'word_length', 'ttr', 'mtld']
matrix = frame[ALL].corr().round(2)
matrix.index = [name.replace('_', ' ').title() for name in matrix.index]
matrix.columns = matrix.index
publish(matrix.reset_index().rename(columns={'index': 'Measure'}),
        'readability_s07_correlations')


,Measure,Fkgl,Fre,Mean Aoa,Response Length,Gunning Fog,Ari,Smog,P90 Aoa,Max Aoa,Difficult Share,Aoa Coverage,Sentence Length,Word Length,Ttr,Mtld
0,Fkgl,1.00,-0.93,0.72,-0.13,0.97,0.96,0.95,0.70,0.28,0.62,-0.36,0.60,0.70,0.30,0.36
1,Fre,-0.93,1.00,-0.87,0.14,-0.91,-0.90,-0.91,-0.82,-0.33,-0.73,0.44,-0.27,-0.89,-0.38,-0.52
2,Mean Aoa,0.72,-0.87,1.00,0.02,0.75,0.71,0.73,0.93,0.48,0.86,-0.37,0.02,0.86,0.22,0.54
3,Response Length,-0.13,0.14,0.02,1.00,-0.07,-0.17,-0.05,0.11,0.52,0.13,0.00,0.00,-0.12,-0.83,-0.14
4,Gunning Fog,0.97,-0.91,0.75,-0.07,1.00,0.93,0.97,0.74,0.31,0.66,-0.39,0.56,0.69,0.25,0.33
5,Ari,0.96,-0.90,0.71,-0.17,0.93,1.00,0.90,0.67,0.25,0.58,-0.36,0.57,0.75,0.37,0.42
6,Smog,0.95,-0.91,0.73,-0.05,0.97,0.90,1.00,0.74,0.33,0.63,-0.38,0.52,0.69,0.23,0.33
7,P90 Aoa,0.70,-0.82,0.93,0.11,0.74,0.67,0.74,1.00,0.51,0.84,-0.35,0.06,0.77,0.12,0.47
8,Max Aoa,0.28,-0.33,0.48,0.52,0.31,0.25,0.33,0.51,1.00,0.51,-0.18,0.05,0.32,-0.40,0.17
9,Difficult Share,0.62,-0.73,0.86,0.13,0.66,0.58,0.63,0.84,0.51,1.00,-0.34,0.03,0.68,0.09,0.41


In [12]:
write_captions()
print(', '.join(f'{count} {kind}' for kind, count
                in sorted(analysis.WRITTEN.items())) + ' written')


2 main table, 7 supplement table written
